In [ ]:
import torch
from dataset import CrossingDataset, collate_fn
from model import MultiTaskNet
from loss import SoftDeltaLoss
from torch.utils.data import DataLoader
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('./eval_results.csv')
df = df.sort_values(by='delta1', ascending=False)
print(df.columns)

columns_main = ['encoder', 'loss', 'teacher', 'weight', 
                    'RMSE', 'RMSE_log', 'AbsRel', 'SqRel', 'delta1', 'delta2', 'delta3', 
                    'mean_iou', 'light_acc', 'kpt_dist',
                    'params_M', 'size_MB', 'GFLOPs', 'latency_ms', 'FPS']

columns_range = ['encoder', '0-5m_RMSE', '0-5m_delta1', '0-5m_delta2', '0-5m_delta3', '0-5m_AbsRel', 
                    '5-10m_RMSE', '5-10m_delta1', '5-10m_delta2', '5-10m_delta3', '5-10m_AbsRel',
                    '10-20m_RMSE', '10-20m_delta1', '10-20m_delta2', '10-20m_delta3', '10-20m_AbsRel']

In [ ]:
df[columns_main].sort_values(by='delta1', ascending=False)

In [ ]:
df[['encoder', 'loss', 'teacher', 'weight', 
        '0-5m_delta1', '5-10m_delta1', '10-20m_delta1', 
        'params_M', 'GFLOPs', 'FPS'
    ]].sort_values(by='5-10m_delta1', ascending=False)

## SI-Log vs SoftDeltaLoss

In [ ]:
df[df['loss'] == 'silog'][columns_main].sort_values(by='delta1', ascending=False)

In [ ]:
df[df['loss'] == 'softdelta'][columns_main].sort_values(by='delta1', ascending=False)

In [ ]:
tmp_df_silog = df[df['loss'] == 'silog'][columns_main]
tmp_df_softdelta = df[df['loss'] == 'softdelta'][columns_main]

encoder_name = tmp_df_silog['encoder'].unique()

In [ ]:
LOWER_IS_BETTER = {"RMSE", "RMSE_log", "AbsRel", "SqRel"}

def get_value(df, encoder, loss, metric, use_teacher, w_teacher):
    """Return the metric value for one (encoder, loss), or np.nan if missing."""
    if use_teacher:
        row = df[(df["encoder"] == encoder) & (df["loss"] == loss) & (df["teacher"] == True) & (df["weight"] == w_teacher)]
    else: 
        row = df[(df["encoder"] == encoder) & (df["loss"] == loss) & (df["teacher"] == False)]
    
    if row.empty:
        return np.nan
    
    return float(row[metric].iloc[0])

def plot_loss_comparison(df, models, metric, use_teacher, w_teacher=None, losses=["silog", "softdelta"], min_ylim=0.5):
    import matplotlib.pyplot as plt
    
    x = np.arange(len(models))
    n = len(losses)
    width = 0.8 / n                      # total group width 0.8

    colors = ["#0E8F8C", "#C63725", "#55A868", "#C44E52"]

    fig, ax = plt.subplots(figsize=(max(7, len(models) * 1.3), 5))
    for i, loss in enumerate(losses):
        vals = [get_value(df, m, loss, metric, use_teacher, w_teacher) for m in models]
        offset = (i - (n - 1) / 2) * width
        bars = ax.bar(x + offset, vals, width, label=loss, color=colors[i % len(colors)], edgecolor="black", linewidth=0.5)
        # value labels on top of each bar
        for b, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(b.get_x() + b.get_width() / 2, v,
                        f"{v:.3f}", ha="center", va="bottom", fontsize=8)

    better = "lower is better" if metric in LOWER_IS_BETTER else "higher is better"
    ax.set_xlabel("Encoder")
    ax.set_ylabel(f"{metric}  ({better})")
    ax.set_title(f"{metric} by encoder: {' vs '.join(losses)} | Teacher: {use_teacher}")
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=10, ha="center")
    ax.legend(title="loss")
    ax.grid(axis="y", alpha=0.3)
    
    # give headroom for the value labels
    valid = [get_value(df, m, l, metric, use_teacher, w_teacher) for m in models for l in losses]
    valid = [v for v in valid if not np.isnan(v)]
    
    if valid:
        ax.set_ylim(min_ylim, max(valid) * 1.15)
    fig.tight_layout()
    
    return fig, ax

In [ ]:
plot_loss_comparison(df, encoder_name, "delta1", use_teacher=False, w_teacher=None, min_ylim=0.6)

In [ ]:
plot_loss_comparison(df, encoder_name, "RMSE", use_teacher=False, w_teacher=None, min_ylim=2.5)

## Solo Vs. With Teacher

In [ ]:
df[(df['encoder'] == 'efficientnet_b0') & (df['loss'] == 'softdelta')][columns_main].loc[2:3]


In [ ]:
from model import MultiTaskNet
from dataset import CrossingDataset, collate_fn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
max_objects = 4
shuffle = True

ds = CrossingDataset(
        'dataset_v2.csv', 
        "test", 
        "new_ds/train/_annotations_coco.json",
        use_teacher=True, 
        max_objects=max_objects)

batch_size = 10
dl = DataLoader(
        ds, 
        batch_size=batch_size, 
        shuffle=shuffle,
        num_workers=0, 
        collate_fn=collate_fn)

LIGHT_NAMES = {0: "No-Light", 1: "Green", 2: "Red"}

batch = next(iter(dl))

In [ ]:
model_b0 = MultiTaskNet(encoder_name='efficientnet_b0', pretrained=False, max_objects=max_objects)
state_b0 = torch.load("./checkpoints/train_efficientnet_b0_softdelta_w0.10_last.pt", map_location=device, weights_only=False)
model_b0.load_state_dict(state_b0, strict=False)
model_b0.to(device)

model_b0_w01 = MultiTaskNet(encoder_name='efficientnet_b0', pretrained=False, max_objects=max_objects)
state_b0_w01 = torch.load("./checkpoints/train_efficientnet_b0_softdelta_no_teacher_last.pt", map_location=device, weights_only=False)
model_b0_w01.load_state_dict(state_b0_w01, strict=False)
model_b0_w01.to(device)

print('Loaded successfully')

In [ ]:
for k in batch:
    if torch.is_tensor(batch[k]):
        batch[k] = batch[k].to(device)
        
out1 = model_b0(batch["image"])
out2 = model_b0_w01(batch["image"])

In [ ]:
import matplotlib.patches as patches

def draw_boxes(ax, bbox, kpts, scores, W, H, box_color, thresh=0.5, show_score=False):
    M = bbox.shape[0]
    for si in range(M):
        if scores[si] < thresh:
            continue
        
        cx, cy, w, h = bbox[si].tolist()
        ax.add_patch(patches.Rectangle(((cx - w / 2) * W, (cy - h / 2) * H), w * W, h * H, fill=False, edgecolor=box_color, lw=2))
        if show_score:
            ax.text((cx - w / 2) * W, (cy - h / 2) * H - 2, f"Obj Exist: {scores[si]:.2f}", color=box_color, fontsize=7, weight="bold")
            
        for j in range(kpts.shape[1]):
            kx, ky = kpts[si, j, 0].item(), kpts[si, j, 1].item()
            v = kpts[si, j, 2].item() if kpts.shape[2] > 2 else 1.0
            if v > 0 and (kx > 0 or ky > 0):
                ax.plot(kx * W, ky * H, "o", color=box_color, ms=5,
                        markeredgecolor="black", markeredgewidth=0.5)

In [ ]:
outs = [out1, out2]
model_names = ["No Teacher", "With Teacher (W:0.1)"]

for i in range(batch_size):
    ncol = 4 + len(outs)          # RGB, RGB+bbox+light, depth GT, + 1 depth per model
    fig, axes = plt.subplots(1, ncol, figsize=(2*ncol, 3.2))

    img = batch["image"][i].permute(1, 2, 0).cpu().numpy().clip(0, 1)
    H, W = img.shape[:2]
    gt_d = batch["depth"][i, 0].cpu().numpy()
    mask = batch["depth_mask"][i, 0].cpu().numpy() > 0.5
    gt_vis = np.where(mask, gt_d, np.nan)
    vmax = np.nanmax(gt_vis) if np.isfinite(gt_vis).any() else 20

    # kolom 0: RGB murni + frame_id
    axes[0].imshow(img)
    axes[0].set_title(batch["frame_id"][i], fontsize=8)
    axes[0].axis("off")

    # kolom 1: RGB + bbox (GT hijau, pred biru) + light
    ax = axes[1]
    ax.imshow(img)
    draw_boxes(ax, batch["bbox"][i].cpu(), batch["keypoints"][i].cpu(), batch["obj_mask"][i].cpu(), W, H, "lime", thresh=0.5)
    
    pred_scores = torch.sigmoid(out1["objectness"][i]).detach().cpu()
    draw_boxes(ax, out1["bbox"][i].detach().cpu(), out1["keypoints_reg"][i].detach().cpu(), pred_scores, W, H, "blue", thresh=0.3, show_score=True)
    
    pl = LIGHT_NAMES[out1["light"][i].argmax().item()]
    gl = LIGHT_NAMES[batch["light"][i].item()]
    ok = "OK" if pl == gl else "X"
    ax.set_title("bbox + light\nGT={} | Pred={} [{}]".format(gl, pl, ok), fontsize=8)
    ax.axis("off")

    # kolom 2: depth GT
    axes[2].imshow(gt_vis, cmap="jet", vmin=0, vmax=vmax)
    axes[2].set_title("Depth GT", fontsize=9)
    axes[2].axis("off")

    # kolom 3: teacher 
    axes[3].imshow(batch["teacher"][i, 0].detach().cpu().numpy(), cmap="jet", vmin=0, vmax=vmax)
    axes[3].set_title("Teacher's Map", fontsize=9)
    axes[3].axis("off")

    # kolom 4..: depth prediksi tiap model (skala sama = vmax GT)
    for j, (out, nm) in enumerate(zip(outs, model_names)):
        pr = out["depth"][i, 0].detach().cpu().numpy()
        axes[4+j].imshow(pr, cmap="jet", vmin=0, vmax=vmax)
        axes[4+j].set_title(f"Pred {nm}", fontsize=9)
        axes[4+j].axis("off")

    plt.tight_layout()
    plt.show()
    
    # break

## Teacher's Impact - Hole Filling Strategy

In [ ]:
df[df['teacher'] == True]

### The impact of choosing weight value

In [ ]:
b0_w01 = df[(df['encoder'] == 'efficientnet_b0') & (df['loss'] == 'softdelta') & (df['teacher'] == True) & (df['weight'] == 0.1)]
b0_w03 = df[(df['encoder'] == 'efficientnet_b0') & (df['loss'] == 'softdelta') & (df['teacher'] == True) & (df['weight'] == 0.3)]
b0_w05 = df[(df['encoder'] == 'efficientnet_b0') & (df['loss'] == 'softdelta') & (df['teacher'] == True) & (df['weight'] == 0.5)]

plt.figure(figsize=(7, 5))
plt.grid(True, axis='y', alpha=0.3)
plt.bar(
    ['0.1', '0.3', '0.5'], 
    [b0_w01['delta1'].values[0], b0_w03['delta1'].values[0], b0_w05['delta1'].values[0]], 
    color=["#B30F0C", "#1E9212", "#173390"])

plt.text(0, b0_w01['delta1'].values[0] + 0.0015, f"{b0_w01['delta1'].values[0]:.3f}", ha="center", va="bottom", fontsize=14, fontweight="bold")
plt.text(1, b0_w03['delta1'].values[0] + 0.0015, f"{b0_w03['delta1'].values[0]:.3f}", ha="center", va="bottom", fontsize=14, fontweight="bold")
plt.text(2, b0_w05['delta1'].values[0] + 0.0015, f"{b0_w05['delta1'].values[0]:.3f}", ha="center", va="bottom", fontsize=14, fontweight="bold")

plt.xlabel("Teacher weight")
plt.ylabel("delta1 (higher is better)")

plt.ylim(0.6, 0.75)

plt.title("Effect of teacher weight for EfficientNet-B0")

In [ ]:
model_b0_w01 = MultiTaskNet(encoder_name='efficientnet_b0', pretrained=False, max_objects=max_objects)
state_b0_w01 = torch.load("./checkpoints/train_efficientnet_b0_softdelta_w0.10_last.pt", map_location=device, weights_only=False)
model_b0_w01.load_state_dict(state_b0_w01, strict=False)
model_b0_w01.to(device)

model_b0_w03 = MultiTaskNet(encoder_name='efficientnet_b0', pretrained=False, max_objects=max_objects)
state_b0_w03 = torch.load("./checkpoints/train_efficientnet_b0_softdelta_w0.30_last.pt", map_location=device, weights_only=False)
model_b0_w03.load_state_dict(state_b0_w03, strict=False)
model_b0_w03.to(device)

model_b0_w05 = MultiTaskNet(encoder_name='efficientnet_b0', pretrained=False, max_objects=max_objects)
state_b0_w05 = torch.load("./checkpoints/train_efficientnet_b0_softdelta_w0.50_last.pt", map_location=device, weights_only=False)
model_b0_w05.load_state_dict(state_b0_w05, strict=False)
model_b0_w05.to(device)

print('Loaded successfully')

In [ ]:
for k in batch:
    if torch.is_tensor(batch[k]):
        batch[k] = batch[k].to(device)
        
out1 = model_b0_w01(batch["image"])
out2 = model_b0_w03(batch["image"])
out3 = model_b0_w05(batch["image"])

In [ ]:
outs = [out1, out2, out3]
model_names = ["w=0.1", "w=0.3", "w=0.5"]

for i in range(batch_size):
    ncol = 4 + len(outs)          # RGB, RGB+bbox+light, depth GT, + 1 depth per model
    fig, axes = plt.subplots(1, ncol, figsize=(2*ncol, 3.2))

    img = batch["image"][i].permute(1, 2, 0).cpu().numpy().clip(0, 1)
    H, W = img.shape[:2]
    gt_d = batch["depth"][i, 0].cpu().numpy()
    mask = batch["depth_mask"][i, 0].cpu().numpy() > 0.5
    gt_vis = np.where(mask, gt_d, np.nan)
    vmax = np.nanmax(gt_vis) if np.isfinite(gt_vis).any() else 20

    # kolom 0: RGB murni + frame_id
    axes[0].imshow(img)
    axes[0].set_title(batch["frame_id"][i], fontsize=8)
    axes[0].axis("off")

    # kolom 1: RGB + bbox (GT hijau, pred biru) + light
    ax = axes[1]
    ax.imshow(img)
    draw_boxes(ax, batch["bbox"][i].cpu(), batch["keypoints"][i].cpu(), batch["obj_mask"][i].cpu(), W, H, "lime", thresh=0.5)
    
    pred_scores = torch.sigmoid(out1["objectness"][i]).detach().cpu()
    draw_boxes(ax, out1["bbox"][i].detach().cpu(), out1["keypoints_reg"][i].detach().cpu(), pred_scores, W, H, "blue", thresh=0.3, show_score=True)
    
    pl = LIGHT_NAMES[out1["light"][i].argmax().item()]
    gl = LIGHT_NAMES[batch["light"][i].item()]
    ok = "OK" if pl == gl else "X"
    ax.set_title("bbox + light\nGT={} | Pred={} [{}]".format(gl, pl, ok), fontsize=8)
    ax.axis("off")

    # kolom 2: depth GT
    axes[2].imshow(gt_vis, cmap="jet", vmin=0, vmax=vmax)
    axes[2].set_title("Depth GT", fontsize=9)
    axes[2].axis("off")

    # kolom 3: teacher 
    axes[3].imshow(batch["teacher"][i, 0].detach().cpu().numpy(), cmap="jet", vmin=0, vmax=vmax)
    axes[3].set_title("Teacher's Map", fontsize=9)
    axes[3].axis("off")

    # kolom 4..: depth prediksi tiap model (skala sama = vmax GT)
    for j, (out, nm) in enumerate(zip(outs, model_names)):
        pr = out["depth"][i, 0].detach().cpu().numpy()
        axes[4+j].imshow(pr, cmap="jet", vmin=0, vmax=vmax)
        axes[4+j].set_title(f"Pred {nm}", fontsize=9)
        axes[4+j].axis("off")

    plt.tight_layout()
    plt.show()
    
    # break

### Model Comparison

In [ ]:
df

In [ ]:
b0_w01 = df[(df['encoder'] == 'efficientnet_b0') & (df['loss'] == 'softdelta') & (df['teacher'] == True) & (df['weight'] == 0.1)]
b0_w03 = df[(df['encoder'] == 'efficientnet_b0') & (df['loss'] == 'softdelta') & (df['teacher'] == True) & (df['weight'] == 0.3)]
b0_w05 = df[(df['encoder'] == 'efficientnet_b0') & (df['loss'] == 'softdelta') & (df['teacher'] == True) & (df['weight'] == 0.5)]

plt.figure(figsize=(7, 5))
plt.grid(True, axis='y', alpha=0.3)
plt.bar(
    ['0.1', '0.3', '0.5'], 
    [b0_w01['delta1'].values[0], b0_w03['delta1'].values[0], b0_w05['delta1'].values[0]], 
    color=["#B30F0C", "#1E9212", "#173390"])

plt.text(0, b0_w01['delta1'].values[0] + 0.0015, f"{b0_w01['delta1'].values[0]:.3f}", ha="center", va="bottom", fontsize=14, fontweight="bold")
plt.text(1, b0_w03['delta1'].values[0] + 0.0015, f"{b0_w03['delta1'].values[0]:.3f}", ha="center", va="bottom", fontsize=14, fontweight="bold")
plt.text(2, b0_w05['delta1'].values[0] + 0.0015, f"{b0_w05['delta1'].values[0]:.3f}", ha="center", va="bottom", fontsize=14, fontweight="bold")

plt.xlabel("Teacher weight")
plt.ylabel("delta1 (higher is better)")

plt.ylim(0.6, 0.75)

plt.title("Effect of teacher weight for EfficientNet-B0")

In [ ]:
enc_name = 'efficientnet_b0'
df_slctd = df[(df['encoder'] == enc_name) & (df['loss'] == 'softdelta')]

df_slctd

In [ ]:
df_slctd['efficientnet_b0']

In [ ]:
names = ['EfficientNet-B0 - No Teacher', 'EfficientNet-B0 Teacher 0.1', 'EfficientNet-B0 Teacher 0.3', 'EfficientNet-B0 Teacher 0.1']
plt.bar(names, df_slctd['delta1'])

In [ ]:
df_slctd

## Teacher's Impact - Train 2 phase

## Zone Evaluation

## Multi-task lain: Deteksi, Keypoint, Light

## Qualitative Evaluation